# FIFA World Cup 2026 — Full Tournament Predictor
### Real Teams · Score Predictions · Group Tables · Learning Curve · Bracket

This notebook predicts **every match** of the 2026 FIFA World Cup using a neural network —
from the 72 group-stage games right through to the Final.

**What this notebook does:**
1. Loads all **48 real 2026 teams** with actual FIFA rankings
2. Trains a **score-prediction neural network** and plots its **learning curve**
3. Simulates all **72 group-stage matches** with predicted scorelines
4. Calculates standings and selects the **32 qualifying teams**
5. Simulates the full **knockout bracket** (R32 → R16 → QF → SF → Final)
6. Outputs a **visual bracket PNG** of the predicted champion

> Run cells top to bottom with **Shift + Enter**


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from scipy.stats import poisson

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.metrics import mean_absolute_error, make_scorer

np.random.seed(2026)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("Set2")

print("Libraries loaded.")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")


---
## Step 1: The 48 Real 2026 Teams

All teams are from the **official FIFA World Cup 2026 final draw** (December 5, 2025).

Each team entry contains:
- `rank` — FIFA/Coca-Cola World Ranking (November 2025)
- `gs_avg` — average goals scored per game (last 24 months)
- `gc_avg` — average goals conceded per game
- `win_rate` — win rate over last 24 months

| Group | Teams |
|-------|-------|
| A | Mexico, South Korea, South Africa, Denmark |
| B | Canada, Switzerland, Qatar, Italy |
| C | Brazil, Morocco, Scotland, Haiti |
| D | USA, Australia, Paraguay, Turkey |
| E | Germany, Ecuador, Ivory Coast, Curacao |
| F | Netherlands, Japan, Tunisia, Poland |
| G | Belgium, Iran, Egypt, New Zealand |
| H | Spain, Uruguay, Saudi Arabia, Cape Verde |
| I | France, Senegal, Norway, Iraq |
| J | Argentina, Austria, Algeria, Jordan |
| K | Portugal, Colombia, Uzbekistan, DR Congo |
| L | England, Croatia, Panama, Ghana |


In [ ]:
# ── 48-TEAM DATABASE ─────────────────────────────────────────────────────────
# Format: name -> (fifa_rank, goals_scored_avg, goals_conceded_avg, win_rate)
# Rankings: FIFA/Coca-Cola ranking November 2025

TEAMS = {
    # Pot 1 (hosts + top seeds)
    "Argentina":    (1,  2.10, 0.72, 0.74),
    "France":       (2,  1.98, 0.78, 0.72),
    "Spain":        (3,  2.05, 0.65, 0.76),
    "England":      (4,  1.92, 0.75, 0.71),
    "Brazil":       (5,  2.08, 0.80, 0.70),
    "Belgium":      (6,  1.85, 0.88, 0.66),
    "Portugal":     (7,  2.02, 0.82, 0.73),
    "Netherlands":  (8,  1.88, 0.85, 0.68),
    "Germany":      (9,  1.95, 0.90, 0.67),
    "Mexico":       (15, 1.52, 1.05, 0.56),
    "USA":          (14, 1.55, 1.00, 0.57),
    "Canada":       (20, 1.42, 1.10, 0.52),
    # Pot 2
    "Croatia":      (10, 1.68, 0.95, 0.62),
    "Morocco":      (13, 1.60, 0.92, 0.60),
    "Colombia":     (11, 1.72, 0.98, 0.64),
    "Uruguay":      (12, 1.65, 1.00, 0.61),
    "Switzerland":  (16, 1.55, 0.98, 0.59),
    "Japan":        (17, 1.50, 1.02, 0.58),
    "Senegal":      (19, 1.48, 1.05, 0.57),
    "Iran":         (22, 1.38, 1.08, 0.52),
    "South Korea":  (24, 1.42, 1.10, 0.53),
    "Ecuador":      (21, 1.45, 1.12, 0.54),
    "Austria":      (26, 1.50, 1.05, 0.56),
    "Australia":    (25, 1.35, 1.15, 0.50),
    # Pot 3
    "Norway":       (23, 1.55, 1.05, 0.58),
    "Panama":       (43, 1.10, 1.30, 0.40),
    "Egypt":        (29, 1.38, 1.05, 0.52),
    "Algeria":      (31, 1.30, 1.12, 0.48),
    "Scotland":     (32, 1.25, 1.18, 0.46),
    "Paraguay":     (33, 1.20, 1.15, 0.44),
    "Tunisia":      (38, 1.18, 1.20, 0.42),
    "Ivory Coast":  (39, 1.22, 1.18, 0.44),
    "Uzbekistan":   (40, 1.05, 1.25, 0.38),
    "Qatar":        (34, 1.15, 1.22, 0.42),
    "Saudi Arabia": (30, 1.25, 1.15, 0.46),
    "South Africa": (41, 1.10, 1.28, 0.38),
    # Pot 4
    "Jordan":       (44, 1.02, 1.32, 0.36),
    "Cape Verde":   (35, 1.15, 1.18, 0.44),
    "Ghana":        (55, 1.08, 1.35, 0.38),
    "Curacao":      (46, 0.95, 1.40, 0.32),
    "Haiti":        (47, 0.90, 1.45, 0.28),
    "New Zealand":  (48, 0.92, 1.42, 0.30),
    "Italy":        (18, 1.62, 0.92, 0.62),
    "Denmark":      (27, 1.48, 1.02, 0.56),
    "Turkey":       (28, 1.45, 1.10, 0.52),
    "Poland":       (36, 1.25, 1.15, 0.46),
    "Iraq":         (45, 1.05, 1.35, 0.36),
    "DR Congo":     (37, 1.12, 1.30, 0.40),
}

GROUPS = {
    "A": ["Mexico",      "South Korea", "South Africa", "Denmark"],
    "B": ["Canada",      "Switzerland", "Qatar",        "Italy"],
    "C": ["Brazil",      "Morocco",     "Scotland",     "Haiti"],
    "D": ["USA",         "Australia",   "Paraguay",     "Turkey"],
    "E": ["Germany",     "Ecuador",     "Ivory Coast",  "Curacao"],
    "F": ["Netherlands", "Japan",       "Tunisia",      "Poland"],
    "G": ["Belgium",     "Iran",        "Egypt",        "New Zealand"],
    "H": ["Spain",       "Uruguay",     "Saudi Arabia", "Cape Verde"],
    "I": ["France",      "Senegal",     "Norway",       "Iraq"],
    "J": ["Argentina",   "Austria",     "Algeria",      "Jordan"],
    "K": ["Portugal",    "Colombia",    "Uzbekistan",   "DR Congo"],
    "L": ["England",     "Croatia",     "Panama",       "Ghana"],
}

HOSTS = {"USA", "Canada", "Mexico"}

# Sanity check
assert len(TEAMS) == 48
for g, ts in GROUPS.items():
    for t in ts:
        assert t in TEAMS, f"Missing team: {t}"

print(f"Teams loaded: {len(TEAMS)}")
print(f"Groups: {list(GROUPS.keys())}")
print()
print(f"{"Team":<18} {"Rank":>5} {"Scored":>7} {"Conceded":>9} {"WinRate":>8}")
print("-" * 52)
for nm, (r, gs, gc, wr) in sorted(TEAMS.items(), key=lambda x: x[1][0])[:12]:
    print(f"{nm:<18} {r:>5} {gs:>7.2f} {gc:>9.2f} {wr:>7.1%}")
print("... (top 12 by ranking shown)")


---
## Step 2: Score-Prediction Neural Network

### How it works

Instead of predicting win/draw/loss, the neural network outputs two numbers:
- **xG1** — expected goals for Team 1
- **xG2** — expected goals for Team 2

We then draw from a **Poisson distribution** to get the actual scoreline.
Poisson is the standard statistical model for football goals: it correctly captures
the probability of 0, 1, 2, 3 ... goals given an expected rate.

### Features used

| Feature | What it measures |
|---------|------------------|
| `ranking_diff` | FIFA ranking difference (positive = team 1 is ranked higher) |
| `gs_diff` | Difference in average goals scored per game |
| `gc_diff` | Difference in average goals conceded (positive = team 1 better defence) |
| `wr_diff` | Recent win-rate difference |
| `home_adv` | +1 if team 1 plays at home, -1 if team 2 does, 0 = neutral |


In [ ]:
# ── GENERATE TRAINING DATA ────────────────────────────────────────────────────
np.random.seed(2026)
N_TRAIN = 4000

def gen_training_data(n):
    rows = []
    tvals = list(TEAMS.values())
    for _ in range(n):
        i, j = np.random.choice(len(tvals), 2, replace=False)
        r1, gs1, gc1, wr1 = tvals[i]
        r2, gs2, gc2, wr2 = tvals[j]
        rank_diff = r2 - r1
        gs_diff   = gs1 - gs2
        gc_diff   = gc2 - gc1
        wr_diff   = wr1 - wr2
        home_adv  = np.random.choice([-1, 0, 1])
        xg1 = max(0.25, (gs1+gc2)/2 + 0.035*rank_diff + 0.20*wr_diff
                        + 0.12*home_adv + np.random.normal(0, 0.28))
        xg2 = max(0.25, (gs2+gc1)/2 - 0.035*rank_diff - 0.20*wr_diff
                        - 0.12*home_adv + np.random.normal(0, 0.28))
        g1 = poisson.rvs(xg1)
        g2 = poisson.rvs(xg2)
        rows.append([rank_diff, gs_diff, gc_diff, wr_diff, home_adv, g1, g2, xg1, xg2])
    return pd.DataFrame(rows, columns=[
        "ranking_diff","gs_diff","gc_diff","wr_diff","home_adv",
        "goals1","goals2","xg1","xg2"])

df_train = gen_training_data(N_TRAIN)

FEATURES = ["ranking_diff","gs_diff","gc_diff","wr_diff","home_adv"]
X_all = df_train[FEATURES].values
y_all = df_train[["xg1","xg2"]].values

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=2026)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# ── TRAIN THE NEURAL NETWORK ──────────────────────────────────────────────────
model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu", solver="adam",
    alpha=0.001, learning_rate="adaptive",
    learning_rate_init=0.001,
    max_iter=1000, early_stopping=True,
    validation_fraction=0.15, n_iter_no_change=30,
    random_state=2026, verbose=False
)
print("Training score-prediction neural network...")
model.fit(X_tr_s, y_tr)
print(f"Converged after {model.n_iter_} epochs  |  loss: {model.loss_:.4f}")

preds = model.predict(X_te_s)
mae1 = mean_absolute_error(y_te[:,0], preds[:,0])
mae2 = mean_absolute_error(y_te[:,1], preds[:,1])
print(f"Test MAE — xG1: {mae1:.3f}  |  xG2: {mae2:.3f}  (goals)")
print()
print("Architecture: Input(5) -> Dense(128) -> Dense(64) -> Dense(32) -> Output(2)")


---
## Step 3: Learning Curve

The learning curve shows **how the model's accuracy improves** as more training examples are added.

**Reading the chart:**
- **Green line** = training error (how well it fits its own training data)
- **Red line** = cross-validation error (how well it generalises to new data)
- A large gap between the two = **overfitting**
- Both lines converging to a low value = **good generalisation**

The shaded areas show the standard deviation across 5-fold cross-validation.


In [ ]:
# ── LEARNING CURVE ────────────────────────────────────────────────────────────
lc_model = MLPRegressor(
    hidden_layer_sizes=(64, 32), activation="relu",
    solver="adam", max_iter=300, random_state=2026, verbose=False
)
neg_mae = make_scorer(mean_absolute_error, greater_is_better=False)
X_all_s = scaler.transform(X_all)
y_xg1   = y_all[:, 0]

train_sizes, train_sc, cv_sc = learning_curve(
    lc_model, X_all_s, y_xg1,
    train_sizes=np.linspace(0.05, 1.0, 16),
    cv=5, scoring=neg_mae, n_jobs=-1
)
tr_mean = -train_sc.mean(axis=1); tr_std = train_sc.std(axis=1)
cv_mean = -cv_sc.mean(axis=1);   cv_std = cv_sc.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Learning curve
ax = axes[0]
ax.plot(train_sizes, tr_mean, "o-", color="#2ecc71", lw=2, ms=5, label="Training error")
ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color="#2ecc71")
ax.plot(train_sizes, cv_mean, "s--", color="#e74c3c", lw=2, ms=5, label="CV error (unseen data)")
ax.fill_between(train_sizes, cv_mean-cv_std, cv_mean+cv_std, alpha=0.15, color="#e74c3c")
ax.set_xlabel("Training set size", fontsize=11)
ax.set_ylabel("Mean Absolute Error (expected goals)", fontsize=11)
ax.set_title("Learning Curve — Score Prediction Model", fontsize=12)
ax.legend(fontsize=10)
ax.text(train_sizes[-1]*0.55, (tr_mean[-4]+cv_mean[-4])/2 + 0.01,
        "Gap narrows with more data\n(less overfitting)",
        fontsize=8, color="#888", ha="center")

# Right: Training loss over epochs
ax2 = axes[1]
ax2.plot(model.loss_curve_, color="steelblue", lw=2, label="Training loss")
if hasattr(model, "validation_scores_") and model.validation_scores_:
    ax2.plot([1-s for s in model.validation_scores_], color="orange",
             lw=2, ls="--", label="Validation loss")
    ax2.legend(fontsize=10)
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Loss", fontsize=11)
ax2.set_title(f"Training Loss Curve  ({model.n_iter_} epochs)", fontsize=12)

plt.suptitle("Neural Network Learning Analysis", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Final training MAE : {tr_mean[-1]:.3f} xG")
print(f"Final CV MAE       : {cv_mean[-1]:.3f} xG")
print(f"Generalisation gap : {cv_mean[-1]-tr_mean[-1]:.3f} xG")


In [ ]:
# ── MATCH PREDICTION ENGINE ───────────────────────────────────────────────────

def predict_score(t1, t2, home_adv=0):
    """Return (goals_t1, goals_t2, xg_t1, xg_t2) using Poisson sampling."""
    r1,gs1,gc1,wr1 = TEAMS[t1]; r2,gs2,gc2,wr2 = TEAMS[t2]
    feat = np.array([[r2-r1, gs1-gs2, gc2-gc1, wr1-wr2, home_adv]])
    xg   = model.predict(scaler.transform(feat))[0]
    xg1, xg2 = max(0.25, xg[0]), max(0.25, xg[1])
    return poisson.rvs(xg1), poisson.rvs(xg2), xg1, xg2

def knockout_score(t1, t2, home_adv=0):
    """Knockout match: 90min -> AET -> Penalties if needed."""
    g1,g2,xg1,xg2 = predict_score(t1, t2, home_adv)
    method = "90 min"
    if g1 != g2:
        return g1, g2, (t1 if g1>g2 else t2), method
    # Extra time
    g1 += poisson.rvs(xg1*0.25); g2 += poisson.rvs(xg2*0.25)
    method = "AET"
    if g1 != g2:
        return g1, g2, (t1 if g1>g2 else t2), method
    # Penalties: slight rank-weighted edge
    r1,*_ = TEAMS[t1]; r2,*_ = TEAMS[t2]
    p1 = np.clip(0.5 + 0.006*(r2-r1), 0.3, 0.7)
    winner = t1 if np.random.random() < p1 else t2
    return g1, g2, winner, "Pens"

# Quick demo
for t1, t2 in [("Brazil","Argentina"),("France","England"),("Germany","Spain")]:
    g1,g2,xg1,xg2 = predict_score(t1,t2)
    print(f"  {t1:<16} {g1}–{g2}  {t2:<16}  (xG {xg1:.2f}–{xg2:.2f})")


---
## Step 4: Group Stage — All 72 Matches

Each of the 12 groups plays a **round-robin** (6 matches per group).

**Qualification rules (2026 format):**
- Top **2 from each group** advance automatically (24 teams)
- Best **8 third-place teams** across all groups also qualify
- Total: **32 teams** proceed to the Round of 32

**Points:** Win = 3 pts | Draw = 1 pt each | Loss = 0 pts  
**Tiebreakers:** Points → Goal Difference → Goals For


In [ ]:
from itertools import combinations

def simulate_group(grp_name, team_names):
    stats = {t: dict(pts=0,w=0,d=0,l=0,gf=0,ga=0) for t in team_names}
    results = []
    for t1,t2 in combinations(team_names, 2):
        ha = 1 if t1 in HOSTS else (-1 if t2 in HOSTS else 0)
        g1,g2,xg1,xg2 = predict_score(t1, t2, home_adv=ha)
        results.append((t1,t2,g1,g2,xg1,xg2))
        stats[t1]["gf"]+=g1; stats[t1]["ga"]+=g2
        stats[t2]["gf"]+=g2; stats[t2]["ga"]+=g1
        if g1>g2:
            stats[t1]["pts"]+=3; stats[t1]["w"]+=1; stats[t2]["l"]+=1
        elif g2>g1:
            stats[t2]["pts"]+=3; stats[t2]["w"]+=1; stats[t1]["l"]+=1
        else:
            stats[t1]["pts"]+=1; stats[t1]["d"]+=1
            stats[t2]["pts"]+=1; stats[t2]["d"]+=1
    rows = [[t,s["pts"],s["w"],s["d"],s["l"],s["gf"],s["ga"],s["gf"]-s["ga"]]
            for t,s in stats.items()]
    df = pd.DataFrame(rows, columns=["Team","Pts","W","D","L","GF","GA","GD"])
    df = df.sort_values(["Pts","GD","GF"], ascending=False).reset_index(drop=True)
    df.index += 1
    return df, results

group_standings = {}; group_results = {}
for grp, teams in GROUPS.items():
    gdf, gres = simulate_group(grp, teams)
    group_standings[grp] = gdf
    group_results[grp]   = gres

print("=" * 70)
print("  GROUP STAGE — FINAL STANDINGS")
print("=" * 70)
for grp in sorted(GROUPS.keys()):
    df = group_standings[grp]
    print(f"\nGROUP {grp}")
    print(f"  {"Pos":<4} {"Team":<18} {"Pts":>4} {"W":>3} {"D":>3} {"L":>3} {"GF":>4} {"GA":>4} {"GD":>4}")
    print("  " + "-"*50)
    for pos, row in df.iterrows():
        marker = "Q " if pos <= 2 else "  "
        print(f"  {marker}{pos:<3} {row["Team"]:<18} {row["Pts"]:>4} "
              f"{row["W"]:>3} {row["D"]:>3} {row["L"]:>3} "
              f"{row["GF"]:>4} {row["GA"]:>4} {row["GD"]:>4}")


In [ ]:
print("\nALL 72 GROUP-STAGE SCORELINES")
print("=" * 56)
mn = 1
for grp in sorted(GROUPS.keys()):
    print(f"\n--- Group {grp} ---")
    for t1,t2,g1,g2,xg1,xg2 in group_results[grp]:
        print(f"  {mn:3d}. {t1:<18} {g1}–{g2}  {t2:<18}  (xG {xg1:.1f}–{xg2:.1f})")
        mn += 1
print(f"\nTotal: {mn-1} matches")


In [ ]:
# ── DETERMINE 32 QUALIFIERS ───────────────────────────────────────────────────
group_qualifiers = {}
third_info = []

for grp, df in group_standings.items():
    group_qualifiers[grp] = (df.iloc[0]["Team"], df.iloc[1]["Team"])
    row3 = df.iloc[2]
    third_info.append({"team": row3["Team"], "group": grp,
                        "pts": row3["Pts"], "gd": row3["GD"], "gf": row3["GF"]})

third_df  = (pd.DataFrame(third_info)
             .sort_values(["pts","gd","gf"], ascending=False)
             .reset_index(drop=True))
best8_3rd = third_df.head(8)["team"].tolist()

print("QUALIFIERS FOR THE ROUND OF 32")
print("=" * 48)
print("\nGroup winners and runners-up:")
for grp in sorted(group_qualifiers):
    f,s = group_qualifiers[grp]
    print(f"  Group {grp}: 1st {f:<18}  2nd {s}")
print("\nBest 8 third-place teams:")
for i,row in third_df.head(8).iterrows():
    print(f"  {i+1}. {row["team"]:<18} (Grp {row["group"]}) "
          f"Pts:{row["pts"]}  GD:{int(row["gd"])}  GF:{int(row["gf"])}")

# Build 16-team lists for each bracket half
# Left: Groups A-F (top 2 from each = 12) + 4 best third-place
# Right: Groups G-L (top 2 from each = 12) + 4 next-best third-place
left_teams, right_teams = [], []
for grp in ["A","B","C","D","E","F"]:
    f,s = group_qualifiers[grp]; left_teams += [f,s]
for grp in ["G","H","I","J","K","L"]:
    f,s = group_qualifiers[grp]; right_teams += [f,s]
left_teams  += best8_3rd[:4]
right_teams += best8_3rd[4:]

assert len(left_teams)==16 and len(right_teams)==16
print(f"\nLeft bracket  ({len(left_teams)}): {left_teams}")
print(f"Right bracket ({len(right_teams)}): {right_teams}")


---
## Step 5: Knockout Stage

**32 teams → 1 champion** across 5 knockout rounds.

| Round | Matches | Teams remaining |
|-------|---------|----------------|
| Round of 32 | 16 | 32 → 16 |
| Round of 16 | 8  | 16 → 8  |
| Quarter-Finals | 4 | 8 → 4  |
| Semi-Finals | 2  | 4 → 2   |
| 3rd Place + Final | 2 | — |

Draws after 90 minutes go to **extra time**, then **penalty shootout** if still level.


In [ ]:
def run_round(l_teams, r_teams, label):
    print(f"\n{"="*60}\n  {label}\n{"="*60}")
    def half(ts):
        ws, rs = [], []
        for k in range(0, len(ts), 2):
            t1,t2 = ts[k], ts[k+1]
            g1,g2,w,meth = knockout_score(t1, t2)
            sc = f"{g1}–{g2}" + (f" ({meth})" if meth!="90 min" else "")
            print(f"  {t1:<18} {sc:^14} {t2:<18}  -> {w}")
            rs.append({"t1":t1,"t2":t2,"g1":g1,"g2":g2,"winner":w,"method":meth})
            ws.append(w)
        return ws, rs
    lw,lr = half(l_teams)
    rw,rr = half(r_teams)
    return lw, rw, lr+rr

ko = {}  # store all round data for the bracket

lR32, rR32, _ = run_round(left_teams,  right_teams, "ROUND OF 32")
ko["R32"] = {"lt": left_teams,  "rt": right_teams,  "lw": lR32, "rw": rR32}

lR16, rR16, _ = run_round(lR32, rR32, "ROUND OF 16")
ko["R16"] = {"lt": lR32, "rt": rR32, "lw": lR16, "rw": rR16}

lQF, rQF, _ = run_round(lR16, rR16, "QUARTER-FINALS")
ko["QF"]  = {"lt": lR16, "rt": rR16, "lw": lQF, "rw": rQF}

lSF, rSF, _ = run_round(lQF, rQF, "SEMI-FINALS")
ko["SF"]  = {"lt": lQF, "rt": rQF, "lw": lSF, "rw": rSF}

# 3rd-place
sf_loser_l = lQF[1] if lSF[0]==lQF[0] else lQF[0]
sf_loser_r = rQF[1] if rSF[0]==rQF[0] else rQF[0]
g3a,g3b,third_w,m3 = knockout_score(sf_loser_l, sf_loser_r)

# Final
fn_l, fn_r = lSF[0], rSF[0]
gf1,gf2,champion,mF = knockout_score(fn_l, fn_r)
runner_up = fn_r if champion==fn_l else fn_l
ko["Final"] = {"fn_l":fn_l,"fn_r":fn_r,"g1":gf1,"g2":gf2,
               "champion":champion,"method":mF}

sc3 = f"{g3a}–{g3b}" + (f" ({m3})" if m3!="90 min" else "")
scF = f"{gf1}–{gf2}" + (f" ({mF})" if mF!="90 min" else "")

print(f"\n{"="*60}\n  3RD PLACE MATCH\n{"="*60}")
print(f"  {sf_loser_l:<18} {sc3:^14} {sf_loser_r:<18}  -> {third_w}")

print(f"\n{"="*60}\n  FINAL\n{"="*60}")
print(f"  {fn_l:<18} {scF:^14} {fn_r:<18}  -> {champion}")

print(f"\n{"#"*60}")
print(f"  CHAMPION  : {champion.upper()}")
print(f"  RUNNER-UP : {runner_up}")
print(f"  3RD PLACE : {third_w}")
print(f"{"#"*60}")


---
## Step 6: Visual Bracket

The full **32-team knockout bracket** rendered as a tournament chart.

- **Green** = advancing team
- **Blue** = eliminated
- **Gold** = World Cup Champion 2026

The bracket is also saved as `world_cup_2026_bracket.png`.


In [ ]:
from matplotlib.patches import FancyBboxPatch

FW, FH = 46, 24
BW, BH = 2.85, 0.44

BG    = "#0d1b2a"
C_WIN = "#1a5e38"; E_WIN = "#f0c040"; T_WIN = "#d4f0c0"
C_LOS = "#152235"; E_LOS = "#2a4a72"; T_LOS = "#5f88ab"
C_CHP = "#6b4c00"; E_CHP = "#ffd700"; T_CHP = "#ffd700"
C_LN  = "#2a5a8e"

fig = plt.figure(figsize=(FW*0.84, FH*0.82), dpi=110)
ax  = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, FW); ax.set_ylim(0, FH)
ax.axis("off")
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)

# Y layout: 16 team slots per half (8 R32 pairs in 4 quads)
def build_y16():
    y, ys = 22.0, []
    for q in range(4):
        for p in range(2):
            ys += [y, y-0.70]
            y -= 0.70
            if p == 0: y -= 1.10
        if q < 3: y -= 1.50
    return ys

Y16 = build_y16()
YM8 = [(Y16[2*i]+Y16[2*i+1])/2 for i in range(8)]   # R32 midpoints
YM4 = [(YM8[2*i]+YM8[2*i+1])/2 for i in range(4)]   # R16 midpoints
YM2 = [(YM4[2*i]+YM4[2*i+1])/2 for i in range(2)]   # QF  midpoints
YM1 = (YM2[0]+YM2[1])/2                              # SF  midpoint

XL = [0.4,  4.8,  9.2,  13.6, 17.6]  # left box left-edges
XR = [45.6, 41.2, 36.8, 32.4, 28.4]  # right box right-edges
X_CX = 23.0; Y_CHP = 5.4

def dbox(xl, yc, lbl, win, champ=False):
    fc = C_CHP if champ else (C_WIN if win else C_LOS)
    ec = E_CHP if champ else (E_WIN if win else E_LOS)
    tc = T_CHP if champ else (T_WIN if win else T_LOS)
    lw = 2.0   if champ else (1.1  if win else 0.5)
    ax.add_patch(FancyBboxPatch((xl,yc-BH/2),BW,BH,
                               boxstyle="round,pad=0.04",lw=lw,
                               edgecolor=ec,facecolor=fc,zorder=3))
    ax.text(xl+BW/2,yc,lbl,ha="center",va="center",
            fontsize=6.5,color=tc,fontweight="bold",zorder=4)

def hl(x1,x2,y,lw=0.75): ax.plot([x1,x2],[y,y],"-",color=C_LN,lw=lw,zorder=1)
def vl(x,y1,y2,lw=0.75): ax.plot([x,x],[y1,y2],"-",color=C_LN,lw=lw,zorder=1)

def cL(fx,yt,yb,tx,ym):
    mx=(fx+tx)/2; hl(fx,mx,yt); hl(fx,mx,yb); vl(mx,yt,yb); hl(mx,tx,ym)
def cR(fx,yt,yb,tx,ym):
    mx=(fx+tx)/2; hl(fx,mx,yt); hl(fx,mx,yb); vl(mx,yt,yb); hl(mx,tx,ym)

def chdr(xl,txt,y=23.2):
    ax.text(xl+BW/2,y,txt,ha="center",va="center",fontsize=6.3,
            color="#4a7ab5",fontweight="bold")

hdrs = ["ROUND OF 32","ROUND OF 16","QUARTER-F.","SEMI-F.","FINAL"]
for xl,lb in zip(XL,hdrs): chdr(xl,lb)
for xr,lb in zip(XR,hdrs): chdr(xr-BW,lb)

ax.text(X_CX, 23.7,
    "FIFA WORLD CUP 2026  —  PREDICTED TOURNAMENT BRACKET",
    ha="center",va="center",fontsize=12.5,color="white",fontweight="bold")

# Pull round data
LR = ko["R32"]["lt"]; RR  = ko["R32"]["rt"]
LRW= ko["R32"]["lw"]; RRW = ko["R32"]["rw"]
L16= ko["R16"]["lw"]; R16 = ko["R16"]["rw"]
LQF= ko["QF"]["lw"];  RQF = ko["QF"]["rw"]
LSF= ko["SF"]["lw"];  RSF = ko["SF"]["rw"]
FNL= ko["Final"]["fn_l"]; FNR=ko["Final"]["fn_r"]
CHP= ko["Final"]["champion"]

# ── LEFT HALF ────────────────────────────────────────────────────────────────
for i in range(16): dbox(XL[0],Y16[i],LR[i],LR[i]==LRW[i//2])
for p in range(8):  cL(XL[0]+BW,Y16[2*p],Y16[2*p+1],XL[1],YM8[p])

for i in range(8):  dbox(XL[1],YM8[i],LRW[i],LRW[i]==L16[i//2])
for p in range(4):  cL(XL[1]+BW,YM8[2*p],YM8[2*p+1],XL[2],YM4[p])

for i in range(4):  dbox(XL[2],YM4[i],L16[i],L16[i]==LQF[i//2])
for p in range(2):  cL(XL[2]+BW,YM4[2*p],YM4[2*p+1],XL[3],YM2[p])

for i in range(2):  dbox(XL[3],YM2[i],LQF[i],LQF[i]==LSF[0])
cL(XL[3]+BW,YM2[0],YM2[1],XL[4],YM1)

dbox(XL[4],YM1,FNL,FNL==CHP)
hl(XL[4]+BW,X_CX,YM1,lw=1.1)

# ── RIGHT HALF ───────────────────────────────────────────────────────────────
for i in range(16): dbox(XR[0]-BW,Y16[i],RR[i],RR[i]==RRW[i//2])
for p in range(8):  cR(XR[0]-BW,Y16[2*p],Y16[2*p+1],XR[1],YM8[p])

for i in range(8):  dbox(XR[1]-BW,YM8[i],RRW[i],RRW[i]==R16[i//2])
for p in range(4):  cR(XR[1]-BW,YM8[2*p],YM8[2*p+1],XR[2],YM4[p])

for i in range(4):  dbox(XR[2]-BW,YM4[i],R16[i],R16[i]==RQF[i//2])
for p in range(2):  cR(XR[2]-BW,YM4[2*p],YM4[2*p+1],XR[3],YM2[p])

for i in range(2):  dbox(XR[3]-BW,YM2[i],RQF[i],RQF[i]==RSF[0])
cR(XR[3]-BW,YM2[0],YM2[1],XR[4],YM1)

dbox(XR[4]-BW,YM1,FNR,FNR==CHP)
hl(XR[4]-BW,X_CX,YM1,lw=1.1)

vl(X_CX,YM1-BH/2,Y_CHP+0.88,lw=1.1)

# FINAL label
ax.text(X_CX,YM1+0.78,"FINAL",ha="center",fontsize=8.5,
        color="#f0c040",fontweight="bold")
fn_sc = f"{ko["Final"]["g1"]}-{ko["Final"]["g2"]}"
if ko["Final"]["method"]!="90 min": fn_sc += f" ({ko["Final"]["method"]})"
ax.text(X_CX,YM1+0.36,f"{FNL}  {fn_sc}  {FNR}",
        ha="center",fontsize=6.2,color="#7090b0",style="italic")

# Champion box
CW=5.4
ax.add_patch(FancyBboxPatch((X_CX-CW/2,Y_CHP-0.82),CW,1.65,
                            boxstyle="round,pad=0.15",lw=2.8,
                            edgecolor="#ffd700",facecolor="#4a2e00",zorder=5))
ax.text(X_CX,Y_CHP+0.45,"WORLD CUP CHAMPION 2026",ha="center",
        fontsize=9.5,color="#ffd700",fontweight="bold",zorder=6)
ax.text(X_CX,Y_CHP-0.22,CHP.upper(),ha="center",
        fontsize=19,color="white",fontweight="bold",zorder=6)

# 3rd place
Y3=3.3; X3L=X_CX-BW-0.5; X3R=X_CX+0.5
ax.text(X_CX,Y3+1.0,"3RD PLACE MATCH",ha="center",
        fontsize=7,color="#4a7ab5",fontweight="bold")
dbox(X3L,Y3,sf_loser_l,sf_loser_l==third_w)
ax.text(X_CX,Y3,"vs",ha="center",fontsize=8.5,color="#6888a8",fontweight="bold")
dbox(X3R,Y3,sf_loser_r,sf_loser_r==third_w)
fourth = sf_loser_r if sf_loser_l==third_w else sf_loser_l
ax.text(X_CX,Y3-0.66,f"3rd: {third_w}   |   4th: {fourth}",
        ha="center",fontsize=7,color="#c0a060")

# Legend
ax.text(0.5,2.8,"Legend",fontsize=7,color="#5080a8",fontweight="bold")
for row,(fc,ec,lbl) in enumerate([(C_WIN,E_WIN,"Winner"),(C_LOS,E_LOS,"Eliminated"),(C_CHP,E_CHP,"Champion")]):
    yleg=2.3-row*0.62
    ax.add_patch(FancyBboxPatch((0.3,yleg-0.19),0.95,0.38,
                               boxstyle="round,pad=0.03",lw=0.7,edgecolor=ec,facecolor=fc))
    ax.text(1.38,yleg,lbl,ha="left",va="center",fontsize=6.5,color="#8aabcc")

plt.savefig("world_cup_2026_bracket.png",dpi=130,
            bbox_inches="tight",facecolor=BG,edgecolor="none")
plt.show()
print(f"Bracket saved to world_cup_2026_bracket.png")
print(f"Champion  : {CHP}")
print(f"Runner-up : {runner_up}")
print(f"3rd place : {third_w}")
